# 1. Generate and inspect expanded optical telemetry
Default: **96 ONTs, 90 days, five-minute samples**. The four-column topology table
is context only; no operational events or topology-based model logic are added.
The simulator uses GPON-inspired optical/FEC semantics, not fitted field parameters.
This notebook inspects training data only. Initial generation may take a few minutes.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
# Resolve relative configured output paths consistently from any notebook.
import os

os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    start + pd.Timedelta(days=settings["generator"]["days"] * f)
    for f in settings["splits"]
]

In [ ]:
native = pd.read_parquet(
    RUN / "telemetry.parquet", filters=[("time", "<", boundaries[0])]
)
topology = pd.read_parquet(RUN / "topology.parquet")
display(topology.head(12))
display(
    topology.groupby(["olt_id", "pon_port_id"]).agg(
        onts=("entity_id", "size"), splitters=("splitter_id", "nunique")
    )
)
display(pd.Series(json.loads((RUN / "generation_checks.json").read_text())["checks"]))
display(native.describe(include="number"))
measurements = [c for c in native if c not in ["time", "device"]]
display(native.groupby("device")[measurements].agg(lambda x: x.isna().mean()).head())

Each ONT has downstream Rx, upstream Rx, ONT Tx and OLT-port Tx, plus ONT/OLT
temperatures, directional pre-FEC BER proxies and directional corrected,
uncorrectable and total codeword counts. FEC fields are **interval counts**, not
cumulative counters. OLT Tx/temperature are shared port measurements repeated per
ONT; do not count them as independent port observations.

The baseline detector still uses downstream Rx only. Compare additional features
through separate experiments before claiming they improve early detection.

In [ ]:
entity = native.device.iloc[0]
view = native.loc[native.device.eq(entity)].set_index("time")
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
view[["rx_dbm", "upstream_rx_dbm"]].plot(ax=axes[0], ylabel="Received power (dBm)")
view[["ont_tx_dbm", "olt_tx_dbm"]].plot(ax=axes[1], ylabel="Transmit power (dBm)")
view[["ont_temperature_c", "olt_temperature_c"]].plot(
    ax=axes[2], ylabel="Temperature (C)"
)
fig.suptitle(f"Training optical telemetry: {entity}")
plt.tight_layout()
plt.show()
ratios = pd.DataFrame(index=view.index)
for direction in ["downstream", "upstream"]:
    total = view[f"{direction}_fec_total_codewords"].where(lambda s: s > 0)
    for kind in ["corrected", "uncorrectable"]:
        ratios[f"{direction}_{kind}"] = (
            view[f"{direction}_fec_{kind}_codewords"] / total
        )
ratios.plot(figsize=(12, 3), ylabel="Fraction of received codewords")
plt.show()
centred = view.rx_dbm - view.rx_dbm.median()
centred.groupby(centred.index.hour).median().plot(
    ylabel="Centred received power (dB)", title="Training daily profile, one ONT"
)
plt.show()